# Gemma4-26B-A4B_Q8_0で画像→text化して、トレーニング用データを作成

LMStudio経由でGemma4-26B-A4Bを使用して、PNG→text→SFT向けテキストを生成

対象データは以下のPDFファイルを1ページ1枚のPNG画像に変換したもの

- https://www.city.bunkyo.lg.jp/documents/3299/r7syogaisyahukusinotebiki_shusei.pdf



### 0. ライブラリインストールabs

In [2]:
!pip install openai tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 30.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [openai]2m3/4 [openai]


### 1. 画像→テキストデータを作成（Thinking On）

In [18]:
prompt_png2text = """
あなたは障碍者福祉の専門家です。
この画像に書かれているテキストをすべて抽出して、マークダウン形式で出力してください。
画像全体のレイアウトや見出しを維持し、文字のみを正確に書き起こしてください。
出力内容に想像を含んではいけません。画像の中に記載されている内容を、そのまま正確に出力してください。
マークダウン以外の情報を出力は絶対に許されません。
"""

In [23]:
import os
import base64
from openai import OpenAI
from pathlib import Path
from tqdm.auto import tqdm

# LM Studioのローカルサーバー設定
# ※ LM Studioアプリ側でローカルサーバーを起動している必要があります
client = OpenAI(
    base_url="http://localhost:1234/v1", # LM Studioのデフォルトポート
    api_key="lm-studio" # APIキーはなんでもOKです
)

# ディレクトリの設定
data_dir = Path("extract_data")
output_dir = Path("extracted_texts_by_gemma4-26B-A4B") # 既存と混ざらないように別フォルダにしています

# 出力先ディレクトリの作成
output_dir.mkdir(parents=True, exist_ok=True)

# 画像をBase64にエンコードする関数
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# extract_data フォルダ内の画像（PNG）リストを取得
# jpgなどの場合は "*.png" を適宜 "*.jpg" や "*.jpeg" などに変更してください
image_files = sorted(list(data_dir.glob("*.png")))
print(f"合計 {len(image_files)} 枚の画像が見つかりました。")

# プログレスバーを表示しながら処理
for image_path in tqdm(image_files, desc="テキスト抽出中"):
    output_filename = image_path.stem + ".md"
    output_file_path = output_dir / output_filename
    
    # すでに抽出済みの場合はスキップ (処理の一時停止・再開に対応)
    if output_file_path.exists():
        continue
        
    try:
        # 画像をBase64に変換
        base64_image = encode_image(image_path)
        
        # LM Studioにリクエストを送信 (Vision API フォーマット)
        response = client.chat.completions.create(
            model="local-model", # LM StudioでロードされているVisionモデルが自動で使われます
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text", 
                            "text": prompt_png2text
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/png;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ],
            temperature=0.0, # 抽出タスクのため、ランダム性を排除
            max_tokens=30000  # 必要に応じて調整してください
        )
        
        # 結果を取得
        extracted_text = response.choices[0].message.content
        
        # Markdownファイルとして保存
        with open(output_file_path, "w", encoding="utf-8") as f:
            f.write(extracted_text)
            
    except Exception as e:
        print(f"\nエラー発生: {image_path.name}")
        print(e)
        
print("処理が完了しました！")


合計 218 枚の画像が見つかりました。


テキスト抽出中:   0%|          | 0/218 [00:00<?, ?it/s]


エラー発生: r7syogaisyahukusinotebiki_shusei_page_023.png
Request timed out.

エラー発生: r7syogaisyahukusinotebiki_shusei_page_076.png
Request timed out.

エラー発生: r7syogaisyahukusinotebiki_shusei_page_080.png
Request timed out.

エラー発生: r7syogaisyahukusinotebiki_shusei_page_162.png
Request timed out.

エラー発生: r7syogaisyahukusinotebiki_shusei_page_200.png
Request timed out.

エラー発生: r7syogaisyahukusinotebiki_shusei_page_201.png
Request timed out.

エラー発生: r7syogaisyahukusinotebiki_shusei_page_202.png
Request timed out.

エラー発生: r7syogaisyahukusinotebiki_shusei_page_217.png
Request timed out.
処理が完了しました！


### 2. 画像→テキストデータを作成（Thinking Off）

In [25]:
import os
import base64
from openai import OpenAI
from pathlib import Path
from tqdm.auto import tqdm

# LM Studioのローカルサーバー設定
# ※ LM Studioアプリ側でローカルサーバーを起動している必要があります
client = OpenAI(
    base_url="http://localhost:1234/v1", # LM Studioのデフォルトポート
    api_key="lm-studio" # APIキーはなんでもOKです
)

# ディレクトリの設定
data_dir = Path("extract_data")
output_dir = Path("extracted_texts_by_gemma4-26B-A4B_thingking_off") # 既存と混ざらないように別フォルダにしています

# 出力先ディレクトリの作成
output_dir.mkdir(parents=True, exist_ok=True)

# 画像をBase64にエンコードする関数
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# extract_data フォルダ内の画像（PNG）リストを取得
# jpgなどの場合は "*.png" を適宜 "*.jpg" や "*.jpeg" などに変更してください
image_files = sorted(list(data_dir.glob("*.png")))
print(f"合計 {len(image_files)} 枚の画像が見つかりました。")

# プログレスバーを表示しながら処理
for image_path in tqdm(image_files, desc="テキスト抽出中"):
    output_filename = image_path.stem + ".md"
    output_file_path = output_dir / output_filename
    
    # すでに抽出済みの場合はスキップ (処理の一時停止・再開に対応)
    if output_file_path.exists():
        continue
        
    try:
        # 画像をBase64に変換
        base64_image = encode_image(image_path)
        
        # LM Studioにリクエストを送信 (Vision API フォーマット)
        response = client.chat.completions.create(
            model="local-model", # LM StudioでロードされているVisionモデルが自動で使われます
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text", 
                            "text": prompt_png2text
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/png;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ],
            temperature=0.0, # 抽出タスクのため、ランダム性を排除
            max_tokens=30000  # 必要に応じて調整してください
        )
        
        # 結果を取得
        extracted_text = response.choices[0].message.content
        
        # Markdownファイルとして保存
        with open(output_file_path, "w", encoding="utf-8") as f:
            f.write(extracted_text)
            
    except Exception as e:
        print(f"\nエラー発生: {image_path.name}")
        print(e)
        
print("処理が完了しました！")


合計 218 枚の画像が見つかりました。


テキスト抽出中:   0%|          | 0/218 [00:00<?, ?it/s]


エラー発生: r7syogaisyahukusinotebiki_shusei_page_063.png
Request timed out.

エラー発生: r7syogaisyahukusinotebiki_shusei_page_096.png
Request timed out.

エラー発生: r7syogaisyahukusinotebiki_shusei_page_123.png
Request timed out.

エラー発生: r7syogaisyahukusinotebiki_shusei_page_187.png
Request timed out.

エラー発生: r7syogaisyahukusinotebiki_shusei_page_194.png
Request timed out.

エラー発生: r7syogaisyahukusinotebiki_shusei_page_195.png
Request timed out.

エラー発生: r7syogaisyahukusinotebiki_shusei_page_196.png
Request timed out.

エラー発生: r7syogaisyahukusinotebiki_shusei_page_202.png
Request timed out.

エラー発生: r7syogaisyahukusinotebiki_shusei_page_205.png
Request timed out.

エラー発生: r7syogaisyahukusinotebiki_shusei_page_211.png
Request timed out.
処理が完了しました！
